# Exp1: Emergence Analysis

In [1]:
"""Experiment 1: ICL Emergence Analysis from Phase 1 Data."""

import typing as t
from pathlib import Path
from dataclasses import dataclass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit


@dataclass
class EmergenceMetrics:
    """Metrics for ICL emergence analysis."""
    
    model_id: str
    config_L: int
    config_m: int
    n_train: int
    emergence_threshold: float
    max_accuracy: float
    context_scaling_slope: float
    baseline_gap: float
    emergence_step: int | None
    training_size_effect: float
    config_complexity_effect: float


class EmergenceAnalyzer:
    """Analyzes ICL emergence patterns from Phase 1 evaluation data."""
    
    def __init__(self, phase1_results_path: Path, output_dir: Path):
        """Initialize emergence analyzer with Phase 1 results."""
        self.phase1_results_path = phase1_results_path
        self.output_dir = output_dir
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.data: pd.DataFrame | None = None
        self.filtered_data: pd.DataFrame | None = None
        self.emergence_metrics: list[EmergenceMetrics] = []
    
    def load_existing_results(self) -> pd.DataFrame:
        """Load Phase 1 evaluation results."""
        if not self.phase1_results_path.exists():
            raise FileNotFoundError(f"Phase 1 results not found: {self.phase1_results_path}")
        
        self.data = pd.read_parquet(self.phase1_results_path)
        print(f"Loaded {len(self.data)} evaluation records from Phase 1")
        
        return self.data
    
    def filter_for_emergence(self) -> pd.DataFrame:
        """Filter data for emergence analysis (within-config only)."""
        if self.data is None:
            raise ValueError("Must load data first")
        
        # Filter for within-config transfer condition and normal control type
        self.filtered_data = self.data[
            (self.data['transfer_condition'] == 'within_config') &
            (self.data['control_type'] == 'normal')
        ].copy()
        
        print(f"Filtered to {len(self.filtered_data)} within-config normal sequences")
        return self.filtered_data
    
    def compute_emergence_metrics(self) -> list[EmergenceMetrics]:
        """Compute emergence metrics for each model."""
        if self.filtered_data is None:
            raise ValueError("Must filter data first")
        
        self.emergence_metrics = []
        
        # Group by model characteristics
        model_groups = self.filtered_data.groupby([
            'model_id', 'config_L', 'config_m', 'n_train', 'checkpoint_step'
        ])
        
        for (model_id, config_L, config_m, n_train, checkpoint_step), group in model_groups:
            metrics = self._compute_single_model_metrics(
                group, model_id, config_L, config_m, n_train
            )
            if metrics:
                self.emergence_metrics.append(metrics)
        
        print(f"Computed emergence metrics for {len(self.emergence_metrics)} models")
        return self.emergence_metrics
    
    def _compute_single_model_metrics(
        self, 
        model_data: pd.DataFrame, 
        model_id: str, 
        config_L: int, 
        config_m: int, 
        n_train: int
    ) -> EmergenceMetrics | None:
        """Compute emergence metrics for a single model."""
        try:
            # Get accuracy by context size
            context_accuracies = model_data.groupby('context_size')['accuracy'].mean().sort_index()
            
            if len(context_accuracies) < 3:  # Need sufficient data points
                return None
            
            # Emergence threshold (first context size > 0.5 accuracy)
            emergence_threshold = self._compute_emergence_threshold(context_accuracies)
            
            # Max accuracy across all context sizes
            max_accuracy = context_accuracies.max()
            
            # Context scaling slope using linear regression
            context_sizes = context_accuracies.index.values
            accuracies = context_accuracies.values
            slope, _, _, _, _ = stats.linregress(context_sizes, accuracies)
            
            # Baseline gap (compare with controls)
            baseline_gap = self._compute_baseline_gap(model_data)
            
            # Emergence step (steepest improvement)
            emergence_step = self._find_emergence_step(context_accuracies)
            
            # Effects (computed later in cross-model analysis)
            training_size_effect = 0.0
            config_complexity_effect = 0.0
            
            return EmergenceMetrics(
                model_id=model_id,
                config_L=config_L,
                config_m=config_m,
                n_train=n_train,
                emergence_threshold=emergence_threshold,
                max_accuracy=max_accuracy,
                context_scaling_slope=slope,
                baseline_gap=baseline_gap,
                emergence_step=emergence_step,
                training_size_effect=training_size_effect,
                config_complexity_effect=config_complexity_effect
            )
            
        except Exception as e:
            print(f"Failed to compute metrics for {model_id}: {e}")
            return None
    
    def _compute_emergence_threshold(self, context_accuracies: pd.Series) -> float:
        """Compute emergence threshold (minimum k for >0.5 accuracy)."""
        threshold = 0.5
        
        for context_size, accuracy in context_accuracies.items():
            if accuracy > threshold:
                return float(context_size)
        
        # No emergence observed
        return float('inf')
    
    def _compute_baseline_gap(self, model_data: pd.DataFrame) -> float:
        """Compute gap between normal and control conditions."""
        if self.data is None:
            return 0.0
        
        # Get control data for same model
        model_id = model_data['model_id'].iloc[0]
        model_controls = self.data[
            (self.data['model_id'] == model_id) &
            (self.data['transfer_condition'] == 'within_config') &
            (self.data['control_type'].isin(['shuffled_context', 'random_context']))
        ]
        
        if len(model_controls) == 0:
            return 0.0
        
        normal_acc = model_data['accuracy'].mean()
        control_acc = model_controls['accuracy'].mean()
        
        return normal_acc - control_acc
    
    def _find_emergence_step(self, context_accuracies: pd.Series) -> int | None:
        """Find context size with steepest accuracy improvement."""
        if len(context_accuracies) < 2:
            return None
        
        improvements = context_accuracies.diff().dropna()
        if len(improvements) == 0:
            return None
        
        max_improvement_idx = improvements.idxmax()
        return int(max_improvement_idx)
    
    def analyze_emergence_patterns(self) -> dict[str, t.Any]:
        """Analyze cross-model emergence patterns."""
        if not self.emergence_metrics:
            raise ValueError("Must compute emergence metrics first")
        
        # Convert to DataFrame for analysis
        metrics_df = pd.DataFrame([
            {
                'model_id': m.model_id,
                'config_L': m.config_L,
                'config_m': m.config_m,
                'n_train': m.n_train,
                'emergence_threshold': m.emergence_threshold,
                'max_accuracy': m.max_accuracy,
                'context_scaling_slope': m.context_scaling_slope,
                'baseline_gap': m.baseline_gap,
                'emergence_step': m.emergence_step
            }
            for m in self.emergence_metrics
        ])
        
        # Analyze patterns
        patterns = {
            'config_effects': self._analyze_config_effects(metrics_df),
            'training_size_effects': self._analyze_training_size_effects(metrics_df),
            'emergence_statistics': self._compute_emergence_statistics(metrics_df),
            'scaling_patterns': self._analyze_scaling_patterns(metrics_df)
        }
        
        return patterns
    
    def _analyze_config_effects(self, metrics_df: pd.DataFrame) -> dict[str, float]:
        """Analyze how configuration affects emergence."""
        config_effects = {}
        
        # Effect of depth (L) on emergence threshold
        if 'config_L' in metrics_df.columns:
            valid_thresholds = metrics_df[metrics_df['emergence_threshold'] != float('inf')]
            if len(valid_thresholds) > 0:
                corr_L = valid_thresholds['config_L'].corr(valid_thresholds['emergence_threshold'])
                config_effects['depth_threshold_correlation'] = corr_L
        
        # Effect of multiplicity (m) on max accuracy
        if 'config_m' in metrics_df.columns:
            corr_m = metrics_df['config_m'].corr(metrics_df['max_accuracy'])
            config_effects['multiplicity_accuracy_correlation'] = corr_m
        
        return config_effects
    
    def _analyze_training_size_effects(self, metrics_df: pd.DataFrame) -> dict[str, float]:
        """Analyze how training size affects emergence."""
        training_effects = {}
        
        # Training size vs emergence threshold
        valid_thresholds = metrics_df[metrics_df['emergence_threshold'] != float('inf')]
        if len(valid_thresholds) > 0:
            corr_train = valid_thresholds['n_train'].corr(valid_thresholds['emergence_threshold'])
            training_effects['training_size_threshold_correlation'] = corr_train
        
        # Training size vs scaling slope
        corr_slope = metrics_df['n_train'].corr(metrics_df['context_scaling_slope'])
        training_effects['training_size_slope_correlation'] = corr_slope
        
        return training_effects
    
    def _compute_emergence_statistics(self, metrics_df: pd.DataFrame) -> dict[str, float]:
        """Compute basic emergence statistics."""
        valid_thresholds = metrics_df[metrics_df['emergence_threshold'] != float('inf')]
        
        stats = {
            'emergence_rate': len(valid_thresholds) / len(metrics_df),
            'mean_emergence_threshold': valid_thresholds['emergence_threshold'].mean() if len(valid_thresholds) > 0 else float('inf'),
            'mean_max_accuracy': metrics_df['max_accuracy'].mean(),
            'mean_baseline_gap': metrics_df['baseline_gap'].mean(),
            'mean_scaling_slope': metrics_df['context_scaling_slope'].mean()
        }
        
        return stats
    
    def _analyze_scaling_patterns(self, metrics_df: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze context scaling patterns."""
        patterns = {
            'positive_slope_rate': (metrics_df['context_scaling_slope'] > 0).mean(),
            'strong_scaling_rate': (metrics_df['context_scaling_slope'] > 0.1).mean(),
            'slope_distribution': metrics_df['context_scaling_slope'].describe().to_dict()
        }
        
        return patterns
    
    def generate_emergence_report(self) -> Path:
        """Generate comprehensive emergence analysis report."""
        report_path = self.output_dir / 'emergence_analysis_report.html'
        
        # Create visualizations
        self._create_emergence_visualizations()
        
        # Generate HTML report
        html_content = self._generate_html_report()
        
        with open(report_path, 'w') as f:
            f.write(html_content)
        
        print(f"Generated emergence report: {report_path}")
        return report_path
    
    def _create_emergence_visualizations(self) -> None:
        """Create emergence analysis visualizations."""
        if not self.emergence_metrics:
            return
        
        # Emergence threshold distribution
        plt.figure(figsize=(12, 8))
        
        plt.subplot(2, 2, 1)
        thresholds = [m.emergence_threshold for m in self.emergence_metrics if m.emergence_threshold != float('inf')]
        plt.hist(thresholds, bins=20, alpha=0.7, edgecolor='black')
        plt.xlabel('Emergence Threshold (Context Size)')
        plt.ylabel('Number of Models')
        plt.title('Distribution of Emergence Thresholds')
        
        # Max accuracy vs config complexity
        plt.subplot(2, 2, 2)
        config_complexity = [m.config_L * m.config_m for m in self.emergence_metrics]
        max_accuracies = [m.max_accuracy for m in self.emergence_metrics]
        plt.scatter(config_complexity, max_accuracies, alpha=0.6)
        plt.xlabel('Configuration Complexity (L × m)')
        plt.ylabel('Max Accuracy')
        plt.title('Max Accuracy vs Configuration Complexity')
        
        # Training size vs emergence threshold
        plt.subplot(2, 2, 3)
        training_sizes = [m.n_train for m in self.emergence_metrics]
        thresholds_for_plot = [m.emergence_threshold if m.emergence_threshold != float('inf') else 10 for m in self.emergence_metrics]
        plt.scatter(training_sizes, thresholds_for_plot, alpha=0.6)
        plt.xlabel('Training Size')
        plt.ylabel('Emergence Threshold')
        plt.title('Training Size vs Emergence Threshold')
        plt.yscale('log')
        
        # Scaling slope distribution
        plt.subplot(2, 2, 4)
        slopes = [m.context_scaling_slope for m in self.emergence_metrics]
        plt.hist(slopes, bins=20, alpha=0.7, edgecolor='black')
        plt.xlabel('Context Scaling Slope')
        plt.ylabel('Number of Models')
        plt.title('Distribution of Context Scaling Slopes')
        
        plt.tight_layout()
        plt.savefig(self.output_dir / 'emergence_analysis.png', dpi=300, bbox_inches='tight')
        plt.close()
    
    def _generate_html_report(self) -> str:
        """Generate HTML report content."""
        return f"""
        <!DOCTYPE html>
        <html>
        <head>
            <title>ICL Emergence Analysis Report</title>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 40px; }}
                .metric {{ background-color: #f5f5f5; padding: 10px; margin: 10px 0; }}
                .visualization {{ text-align: center; margin: 20px 0; }}
            </style>
        </head>
        <body>
            <h1>ICL Emergence Analysis Report</h1>
            <h2>Summary</h2>
            <div class="metric">Total Models Analyzed: {len(self.emergence_metrics)}</div>
            
            <h2>Emergence Visualizations</h2>
            <div class="visualization">
                <img src="emergence_analysis.png" alt="Emergence Analysis Plots" style="max-width: 100%;">
            </div>
            
            <h2>Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</h2>
        </body>
        </html>
        """
    
    def validate_data_completeness(self) -> dict[str, t.Any]:
        """Validate completeness of Phase 1 data for emergence analysis."""
        if self.data is None or self.filtered_data is None:
            raise ValueError("Must load and filter data first")
        
        validation = {
            'total_records': len(self.data),
            'within_config_records': len(self.filtered_data),
            'unique_models': self.filtered_data['model_id'].nunique(),
            'context_sizes_available': sorted(self.filtered_data['context_size'].unique()),
            'configs_available': len(self.filtered_data.groupby(['config_L', 'config_m'])),
            'missing_context_sizes': [],
            'incomplete_models': []
        }
        
        # Check for expected context sizes
        expected_context_sizes = [1, 2, 3, 4, 5, 6, 8]
        available_context_sizes = set(self.filtered_data['context_size'].unique())
        missing = [k for k in expected_context_sizes if k not in available_context_sizes]
        validation['missing_context_sizes'] = missing
        
        # Check model completeness
        for model_id in self.filtered_data['model_id'].unique():
            model_data = self.filtered_data[self.filtered_data['model_id'] == model_id]
            model_context_sizes = set(model_data['context_size'].unique())
            if not set(expected_context_sizes).issubset(model_context_sizes):
                validation['incomplete_models'].append(model_id)
        
        return validation
    
    def save_emergence_metrics(self) -> Path:
        """Save emergence metrics to CSV."""
        if not self.emergence_metrics:
            raise ValueError("No emergence metrics to save")
        
        metrics_df = pd.DataFrame([
            {
                'model_id': m.model_id,
                'config_L': m.config_L,
                'config_m': m.config_m,
                'n_train': m.n_train,
                'emergence_threshold': m.emergence_threshold,
                'max_accuracy': m.max_accuracy,
                'context_scaling_slope': m.context_scaling_slope,
                'baseline_gap': m.baseline_gap,
                'emergence_step': m.emergence_step,
                'training_size_effect': m.training_size_effect,
                'config_complexity_effect': m.config_complexity_effect
            }
            for m in self.emergence_metrics
        ])
        
        output_path = self.output_dir / 'emergence_metrics.csv'
        metrics_df.to_csv(output_path, index=False)
        
        print(f"Saved emergence metrics: {output_path}")
        return output_path


def run_emergence_analysis(
    phase1_results_path: str | Path,
    output_dir: str | Path
) -> dict[str, t.Any]:
    """Run complete emergence analysis pipeline."""
    analyzer = EmergenceAnalyzer(Path(phase1_results_path), Path(output_dir))
    
    # Load and process data
    analyzer.load_existing_results()
    analyzer.filter_for_emergence()
    
    # Validate data completeness
    validation = analyzer.validate_data_completeness()
    print(f"Data validation: {validation['within_config_records']} records, {validation['unique_models']} models")
    
    # Compute metrics and analyze patterns
    analyzer.compute_emergence_metrics()
    patterns = analyzer.analyze_emergence_patterns()
    
    # Generate outputs
    metrics_path = analyzer.save_emergence_metrics()
    report_path = analyzer.generate_emergence_report()
    
    return {
        'emergence_metrics_path': metrics_path,
        'report_path': report_path,
        'patterns': patterns,
        'validation': validation,
        'total_models_analyzed': len(analyzer.emergence_metrics)
    }



In [5]:

# Test function
def test_emergence_analysis() -> None:
    """Test emergence analysis with sample data."""
    # This would use actual Phase 1 results path
    phase1_path = Path("/Users/jliu/workspace/ICL/results/raw/raw_evaluations/icl_performance.parquet")
    output_dir = Path("/Users/jliu/workspace/ICL/results/emergence_analysis_results")
    
    if phase1_path.exists():
        results = run_emergence_analysis(phase1_path, output_dir)
        print(f"Emergence analysis completed: {results['total_models_analyzed']} models")
    else:
        print("Phase 1 results not found - run Phase 1 evaluation first")


if __name__ == "__main__":
    test_emergence_analysis()

Loaded 6426 evaluation records from Phase 1
Filtered to 342 within-config normal sequences
Data validation: 342 records, 6 models
Computed emergence metrics for 6 models
Saved emergence metrics: /Users/jliu/workspace/ICL/results/emergence_analysis_results/emergence_metrics.csv
Generated emergence report: /Users/jliu/workspace/ICL/results/emergence_analysis_results/emergence_analysis_report.html
Emergence analysis completed: 6 models


# Exp 2: transfer 

In [3]:
"""Experiment 2: Transfer Learning Analysis from Phase 1 Data."""

import typing as t
from pathlib import Path
from dataclasses import dataclass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


@dataclass
class TransferMetrics:
    """Metrics for transfer learning analysis."""
    
    model_id: str
    train_config_L: int
    train_config_m: int
    test_config_L: int
    test_config_m: int
    n_train: int
    within_config_accuracy: float
    transfer_accuracy: float
    transfer_degradation: float
    transfer_type: str  # "depth", "synonym", "full"
    depth_increase: int
    synonym_increase: int
    transfer_success: bool
    transfer_scaling_slope: float


class TransferAnalyzer:
    """Analyzes ICL transfer learning patterns from Phase 1 evaluation data."""
    
    def __init__(self, phase1_results_path: Path, output_dir: Path):
        """Initialize transfer analyzer with Phase 1 results."""
        self.phase1_results_path = phase1_results_path
        self.output_dir = output_dir
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.data: pd.DataFrame | None = None
        self.transfer_pairs: pd.DataFrame | None = None
        self.transfer_metrics: list[TransferMetrics] = []
        self.transfer_matrix: pd.DataFrame | None = None
    
    def load_existing_results(self) -> pd.DataFrame:
        """Load Phase 1 evaluation results."""
        if not self.phase1_results_path.exists():
            raise FileNotFoundError(f"Phase 1 results not found: {self.phase1_results_path}")
        
        self.data = pd.read_parquet(self.phase1_results_path)
        print(f"Loaded {len(self.data)} evaluation records from Phase 1")
        
        # Filter for normal control type only
        self.data = self.data[self.data['control_type'] == 'normal'].copy()
        print(f"Filtered to {len(self.data)} normal control sequences")
        
        return self.data
    
    def create_transfer_pairs(self) -> pd.DataFrame:
        """Create within-config vs transfer performance pairs."""
        if self.data is None:
            raise ValueError("Must load data first")
        
        # Separate within-config and transfer conditions
        within_config = self.data[self.data['transfer_condition'] == 'within_config'].copy()
        transfer_conditions = self.data[self.data['transfer_condition'] != 'within_config'].copy()
        
        print(f"Within-config records: {len(within_config)}")
        print(f"Transfer records: {len(transfer_conditions)}")
        
        # Create pairs by matching models and context sizes
        pairs = []
        
        for _, transfer_row in transfer_conditions.iterrows():
            # Find matching within-config performance
            matching_within = within_config[
                (within_config['model_id'] == transfer_row['model_id']) &
                (within_config['context_size'] == transfer_row['context_size']) &
                (within_config['config_L'] == transfer_row['config_L']) &
                (within_config['config_m'] == transfer_row['config_m'])
            ]
            
            if len(matching_within) > 0:
                within_row = matching_within.iloc[0]  # Take first match
                
                pair = {
                    'model_id': transfer_row['model_id'],
                    'train_config_L': transfer_row['config_L'],
                    'train_config_m': transfer_row['config_m'],
                    'test_config_L': transfer_row['target_config_L'],
                    'test_config_m': transfer_row['target_config_m'],
                    'n_train': transfer_row['n_train'],
                    'context_size': transfer_row['context_size'],
                    'within_accuracy': within_row['accuracy'],
                    'transfer_accuracy': transfer_row['accuracy'],
                    'transfer_condition': transfer_row['transfer_condition'],
                    'checkpoint_step': transfer_row['checkpoint_step']
                }
                pairs.append(pair)
        
        self.transfer_pairs = pd.DataFrame(pairs)
        print(f"Created {len(self.transfer_pairs)} transfer pairs")
        
        return self.transfer_pairs
    
    def compute_transfer_metrics(self) -> list[TransferMetrics]:
        """Compute transfer degradation metrics for each transfer pair."""
        if self.transfer_pairs is None:
            raise ValueError("Must create transfer pairs first")
        
        self.transfer_metrics = []
        
        for _, pair in self.transfer_pairs.iterrows():
            metrics = self._compute_single_transfer_metrics(pair)
            if metrics:
                self.transfer_metrics.append(metrics)
        
        print(f"Computed transfer metrics for {len(self.transfer_metrics)} transfer pairs")
        return self.transfer_metrics
    
    def _compute_single_transfer_metrics(self, pair: pd.Series) -> TransferMetrics | None:
        """Compute transfer metrics for a single transfer pair."""
        try:
            # Basic transfer metrics
            transfer_degradation = pair['within_accuracy'] - pair['transfer_accuracy']
            transfer_success = pair['transfer_accuracy'] > 0.5  # Threshold for success
            
            # Determine transfer type and increases
            train_L, train_m = pair['train_config_L'], pair['train_config_m']
            test_L, test_m = pair['test_config_L'], pair['test_config_m']
            
            depth_increase = test_L - train_L
            synonym_increase = test_m - train_m
            
            transfer_type = self._classify_transfer_type(
                pair['transfer_condition'], depth_increase, synonym_increase
            )
            
            # Compute transfer scaling slope (accuracy vs context size for this transfer)
            transfer_scaling_slope = self._compute_transfer_scaling_slope(
                pair['model_id'], train_L, train_m, test_L, test_m
            )
            
            return TransferMetrics(
                model_id=pair['model_id'],
                train_config_L=train_L,
                train_config_m=train_m,
                test_config_L=test_L,
                test_config_m=test_m,
                n_train=pair['n_train'],
                within_config_accuracy=pair['within_accuracy'],
                transfer_accuracy=pair['transfer_accuracy'],
                transfer_degradation=transfer_degradation,
                transfer_type=transfer_type,
                depth_increase=depth_increase,
                synonym_increase=synonym_increase,
                transfer_success=transfer_success,
                transfer_scaling_slope=transfer_scaling_slope
            )
            
        except Exception as e:
            print(f"Failed to compute transfer metrics for pair: {e}")
            return None
    
    def _classify_transfer_type(
        self, transfer_condition: str, depth_increase: int, synonym_increase: int
    ) -> str:
        """Classify transfer type based on condition and configuration changes."""
        if transfer_condition == 'cross_L' or (depth_increase > 0 and synonym_increase == 0):
            return 'depth'
        elif transfer_condition == 'cross_m' or (depth_increase == 0 and synonym_increase > 0):
            return 'synonym'
        elif transfer_condition == 'cross_config' or (depth_increase > 0 and synonym_increase > 0):
            return 'full'
        else:
            return 'unknown'
    
    def _compute_transfer_scaling_slope(
        self, model_id: str, train_L: int, train_m: int, test_L: int, test_m: int
    ) -> float:
        """Compute transfer scaling slope across context sizes."""
        if self.data is None:
            return 0.0
        
        # Get transfer data for this specific model and config pair
        transfer_data = self.data[
            (self.data['model_id'] == model_id) &
            (self.data['config_L'] == train_L) &
            (self.data['config_m'] == train_m) &
            (self.data['target_config_L'] == test_L) &
            (self.data['target_config_m'] == test_m) &
            (self.data['transfer_condition'] != 'within_config')
        ]
        
        if len(transfer_data) < 3:  # Need sufficient points
            return 0.0
        
        # Compute slope across context sizes
        context_accuracies = transfer_data.groupby('context_size')['accuracy'].mean()
        
        if len(context_accuracies) < 2:
            return 0.0
        
        context_sizes = context_accuracies.index.values
        accuracies = context_accuracies.values
        
        try:
            slope, _, _, _, _ = stats.linregress(context_sizes, accuracies)
            return slope
        except:
            return 0.0
    
    def analyze_transfer_patterns(self) -> dict[str, t.Any]:
        """Analyze transfer patterns across different transfer types."""
        if not self.transfer_metrics:
            raise ValueError("Must compute transfer metrics first")
        
        # Convert to DataFrame for analysis
        metrics_df = pd.DataFrame([
            {
                'model_id': m.model_id,
                'train_config_L': m.train_config_L,
                'train_config_m': m.train_config_m,
                'test_config_L': m.test_config_L,
                'test_config_m': m.test_config_m,
                'n_train': m.n_train,
                'within_config_accuracy': m.within_config_accuracy,
                'transfer_accuracy': m.transfer_accuracy,
                'transfer_degradation': m.transfer_degradation,
                'transfer_type': m.transfer_type,
                'depth_increase': m.depth_increase,
                'synonym_increase': m.synonym_increase,
                'transfer_success': m.transfer_success,
                'transfer_scaling_slope': m.transfer_scaling_slope
            }
            for m in self.transfer_metrics
        ])
        
        patterns = {
            'transfer_type_analysis': self._analyze_transfer_types(metrics_df),
            'degradation_patterns': self._analyze_degradation_patterns(metrics_df),
            'success_rates': self._analyze_success_rates(metrics_df),
            'scaling_effects': self._analyze_scaling_effects(metrics_df),
            'config_effects': self._analyze_config_transfer_effects(metrics_df)
        }
        
        return patterns
    
    def _analyze_transfer_types(self, metrics_df: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze performance by transfer type."""
        type_analysis = {}
        
        for transfer_type in ['depth', 'synonym', 'full']:
            type_data = metrics_df[metrics_df['transfer_type'] == transfer_type]
            
            if len(type_data) > 0:
                type_analysis[transfer_type] = {
                    'count': len(type_data),
                    'mean_degradation': type_data['transfer_degradation'].mean(),
                    'success_rate': type_data['transfer_success'].mean(),
                    'mean_within_accuracy': type_data['within_config_accuracy'].mean(),
                    'mean_transfer_accuracy': type_data['transfer_accuracy'].mean(),
                    'degradation_std': type_data['transfer_degradation'].std()
                }
        
        return type_analysis
    
    def _analyze_degradation_patterns(self, metrics_df: pd.DataFrame) -> dict[str, float]:
        """Analyze transfer degradation patterns."""
        patterns = {
            'overall_mean_degradation': metrics_df['transfer_degradation'].mean(),
            'overall_degradation_std': metrics_df['transfer_degradation'].std(),
            'positive_degradation_rate': (metrics_df['transfer_degradation'] > 0).mean(),
            'severe_degradation_rate': (metrics_df['transfer_degradation'] > 0.2).mean(),
            'improvement_rate': (metrics_df['transfer_degradation'] < 0).mean()
        }
        
        return patterns
    
    def _analyze_success_rates(self, metrics_df: pd.DataFrame) -> dict[str, float]:
        """Analyze transfer success rates."""
        success_rates = {
            'overall_success_rate': metrics_df['transfer_success'].mean(),
            'success_by_training_size': {},
            'success_by_depth_increase': {},
            'success_by_synonym_increase': {}
        }
        
        # Success by training size
        for n_train in sorted(metrics_df['n_train'].unique()):
            train_data = metrics_df[metrics_df['n_train'] == n_train]
            success_rates['success_by_training_size'][n_train] = train_data['transfer_success'].mean()
        
        # Success by depth increase
        for depth_inc in sorted(metrics_df['depth_increase'].unique()):
            depth_data = metrics_df[metrics_df['depth_increase'] == depth_inc]
            if len(depth_data) > 0:
                success_rates['success_by_depth_increase'][depth_inc] = depth_data['transfer_success'].mean()
        
        # Success by synonym increase
        for syn_inc in sorted(metrics_df['synonym_increase'].unique()):
            syn_data = metrics_df[metrics_df['synonym_increase'] == syn_inc]
            if len(syn_data) > 0:
                success_rates['success_by_synonym_increase'][syn_inc] = syn_data['transfer_success'].mean()
        
        return success_rates
    
    def _analyze_scaling_effects(self, metrics_df: pd.DataFrame) -> dict[str, float]:
        """Analyze transfer scaling effects."""
        scaling_effects = {
            'mean_transfer_scaling_slope': metrics_df['transfer_scaling_slope'].mean(),
            'positive_scaling_rate': (metrics_df['transfer_scaling_slope'] > 0).mean(),
            'strong_scaling_rate': (metrics_df['transfer_scaling_slope'] > 0.05).mean(),
            'scaling_vs_degradation_correlation': metrics_df['transfer_scaling_slope'].corr(
                metrics_df['transfer_degradation']
            )
        }
        
        return scaling_effects
    
    def _analyze_config_transfer_effects(self, metrics_df: pd.DataFrame) -> dict[str, float]:
        """Analyze how configuration differences affect transfer."""
        config_effects = {}
        
        # Effect of depth increase on degradation
        depth_corr = metrics_df['depth_increase'].corr(metrics_df['transfer_degradation'])
        config_effects['depth_increase_degradation_correlation'] = depth_corr
        
        # Effect of synonym increase on degradation
        syn_corr = metrics_df['synonym_increase'].corr(metrics_df['transfer_degradation'])
        config_effects['synonym_increase_degradation_correlation'] = syn_corr
        
        # Effect of total complexity change
        metrics_df['complexity_change'] = (
            metrics_df['test_config_L'] * metrics_df['test_config_m'] -
            metrics_df['train_config_L'] * metrics_df['train_config_m']
        )
        complexity_corr = metrics_df['complexity_change'].corr(metrics_df['transfer_degradation'])
        config_effects['complexity_change_degradation_correlation'] = complexity_corr
        
        return config_effects
    
    def create_transfer_matrix(self) -> pd.DataFrame:
        """Create train→test configuration transfer performance matrix."""
        if not self.transfer_metrics:
            raise ValueError("Must compute transfer metrics first")
        
        # Create matrix with train configs as rows, test configs as columns
        matrix_data = []
        
        for metric in self.transfer_metrics:
            train_config = f"({metric.train_config_L}, {metric.train_config_m})"
            test_config = f"({metric.test_config_L}, {metric.test_config_m})"
            
            matrix_data.append({
                'train_config': train_config,
                'test_config': test_config,
                'transfer_accuracy': metric.transfer_accuracy,
                'transfer_degradation': metric.transfer_degradation,
                'n_samples': 1
            })
        
        matrix_df = pd.DataFrame(matrix_data)
        
        # Aggregate by config pairs (average across models)
        aggregated = matrix_df.groupby(['train_config', 'test_config']).agg({
            'transfer_accuracy': 'mean',
            'transfer_degradation': 'mean',
            'n_samples': 'sum'
        }).reset_index()
        
        # Pivot to create transfer matrix
        transfer_accuracy_matrix = aggregated.pivot(
            index='train_config', 
            columns='test_config', 
            values='transfer_accuracy'
        )
        
        transfer_degradation_matrix = aggregated.pivot(
            index='train_config',
            columns='test_config', 
            values='transfer_degradation'
        )
        
        self.transfer_matrix = {
            'accuracy': transfer_accuracy_matrix,
            'degradation': transfer_degradation_matrix,
            'raw_data': aggregated
        }
        
        print(f"Created transfer matrix: {transfer_accuracy_matrix.shape}")
        return transfer_accuracy_matrix
    
    def generate_transfer_report(self) -> Path:
        """Generate comprehensive transfer analysis report."""
        report_path = self.output_dir / 'transfer_analysis_report.html'
        
        # Create visualizations
        self._create_transfer_visualizations()
        
        # Generate HTML report
        html_content = self._generate_html_report()
        
        with open(report_path, 'w') as f:
            f.write(html_content)
        
        print(f"Generated transfer report: {report_path}")
        return report_path
    
    def _create_transfer_visualizations(self) -> None:
        """Create transfer analysis visualizations."""
        if not self.transfer_metrics:
            return
        
        # Create comprehensive visualization grid
        plt.figure(figsize=(16, 12))
        
        # 1. Transfer degradation by type
        plt.subplot(3, 3, 1)
        transfer_types = [m.transfer_type for m in self.transfer_metrics]
        degradations = [m.transfer_degradation for m in self.transfer_metrics]
        
        type_degradations = {}
        for t_type in ['depth', 'synonym', 'full']:
            type_degradations[t_type] = [d for m, d in zip(transfer_types, degradations) if m == t_type]
        
        plt.boxplot([type_degradations[t] for t in ['depth', 'synonym', 'full']], 
                   labels=['Depth', 'Synonym', 'Full'])
        plt.ylabel('Transfer Degradation')
        plt.title('Transfer Degradation by Type')
        
        # 2. Transfer accuracy vs within-config accuracy
        plt.subplot(3, 3, 2)
        within_accs = [m.within_config_accuracy for m in self.transfer_metrics]
        transfer_accs = [m.transfer_accuracy for m in self.transfer_metrics]
        plt.scatter(within_accs, transfer_accs, alpha=0.6)
        plt.plot([0, 1], [0, 1], 'r--', alpha=0.8)  # Perfect transfer line
        plt.xlabel('Within-Config Accuracy')
        plt.ylabel('Transfer Accuracy')
        plt.title('Transfer vs Within-Config Performance')
        
        # 3. Degradation vs depth increase
        plt.subplot(3, 3, 3)
        depth_increases = [m.depth_increase for m in self.transfer_metrics]
        plt.scatter(depth_increases, degradations, alpha=0.6)
        plt.xlabel('Depth Increase (ΔL)')
        plt.ylabel('Transfer Degradation')
        plt.title('Degradation vs Depth Increase')
        
        # 4. Degradation vs synonym increase
        plt.subplot(3, 3, 4)
        synonym_increases = [m.synonym_increase for m in self.transfer_metrics]
        plt.scatter(synonym_increases, degradations, alpha=0.6)
        plt.xlabel('Synonym Increase (Δm)')
        plt.ylabel('Transfer Degradation')
        plt.title('Degradation vs Synonym Increase')
        
        # 5. Success rate by training size
        plt.subplot(3, 3, 5)
        training_sizes = sorted(set(m.n_train for m in self.transfer_metrics))
        success_rates = []
        for size in training_sizes:
            size_metrics = [m for m in self.transfer_metrics if m.n_train == size]
            success_rate = sum(m.transfer_success for m in size_metrics) / len(size_metrics)
            success_rates.append(success_rate)
        
        plt.plot(training_sizes, success_rates, 'o-')
        plt.xlabel('Training Size')
        plt.ylabel('Transfer Success Rate')
        plt.title('Success Rate vs Training Size')
        plt.xscale('log')
        
        # 6. Transfer matrix heatmap (if available)
        if self.transfer_matrix is not None:
            plt.subplot(3, 3, 6)
            sns.heatmap(self.transfer_matrix['accuracy'], annot=True, fmt='.2f', cmap='viridis')
            plt.title('Transfer Accuracy Matrix')
            plt.xlabel('Test Config')
            plt.ylabel('Train Config')
        
        # 7. Degradation distribution
        plt.subplot(3, 3, 7)
        plt.hist(degradations, bins=20, alpha=0.7, edgecolor='black')
        plt.xlabel('Transfer Degradation')
        plt.ylabel('Count')
        plt.title('Transfer Degradation Distribution')
        
        # 8. Transfer scaling slopes
        plt.subplot(3, 3, 8)
        scaling_slopes = [m.transfer_scaling_slope for m in self.transfer_metrics]
        plt.hist(scaling_slopes, bins=20, alpha=0.7, edgecolor='black')
        plt.xlabel('Transfer Scaling Slope')
        plt.ylabel('Count')
        plt.title('Transfer Scaling Slope Distribution')
        
        # 9. Success rate by transfer type
        plt.subplot(3, 3, 9)
        type_success_rates = {}
        for t_type in ['depth', 'synonym', 'full']:
            type_metrics = [m for m in self.transfer_metrics if m.transfer_type == t_type]
            if type_metrics:
                type_success_rates[t_type] = sum(m.transfer_success for m in type_metrics) / len(type_metrics)
        
        plt.bar(type_success_rates.keys(), type_success_rates.values())
        plt.ylabel('Success Rate')
        plt.title('Success Rate by Transfer Type')
        
        plt.tight_layout()
        plt.savefig(self.output_dir / 'transfer_analysis.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        # Create transfer matrix visualization separately
        if self.transfer_matrix is not None:
            plt.figure(figsize=(12, 8))
            
            plt.subplot(1, 2, 1)
            sns.heatmap(self.transfer_matrix['accuracy'], annot=True, fmt='.2f', 
                       cmap='viridis', cbar_kws={'label': 'Transfer Accuracy'})
            plt.title('Transfer Accuracy Matrix')
            plt.xlabel('Test Configuration')
            plt.ylabel('Train Configuration')
            
            plt.subplot(1, 2, 2)
            sns.heatmap(self.transfer_matrix['degradation'], annot=True, fmt='.2f',
                       cmap='RdYlBu_r', cbar_kws={'label': 'Transfer Degradation'})
            plt.title('Transfer Degradation Matrix')
            plt.xlabel('Test Configuration')
            plt.ylabel('Train Configuration')
            
            plt.tight_layout()
            plt.savefig(self.output_dir / 'transfer_matrix.png', dpi=300, bbox_inches='tight')
            plt.close()
    
    def _generate_html_report(self) -> str:
        """Generate HTML report content."""
        return f"""
        <!DOCTYPE html>
        <html>
        <head>
            <title>ICL Transfer Learning Analysis Report</title>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 40px; }}
                .metric {{ background-color: #f5f5f5; padding: 10px; margin: 10px 0; }}
                .visualization {{ text-align: center; margin: 20px 0; }}
            </style>
        </head>
        <body>
            <h1>ICL Transfer Learning Analysis Report</h1>
            <h2>Summary</h2>
            <div class="metric">Total Transfer Pairs Analyzed: {len(self.transfer_metrics)}</div>
            
            <h2>Transfer Analysis Visualizations</h2>
            <div class="visualization">
                <img src="transfer_analysis.png" alt="Transfer Analysis Plots" style="max-width: 100%;">
            </div>
            
            <h2>Transfer Matrix</h2>
            <div class="visualization">
                <img src="transfer_matrix.png" alt="Transfer Matrix" style="max-width: 100%;">
            </div>
            
            <h2>Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</h2>
        </body>
        </html>
        """
    
    def validate_transfer_completeness(self) -> dict[str, t.Any]:
        """Validate completeness of Phase 1 data for transfer analysis."""
        if self.data is None:
            raise ValueError("Must load data first")
        
        validation = {
            'total_records': len(self.data),
            'transfer_conditions_available': sorted(self.data['transfer_condition'].unique()),
            'within_config_records': len(self.data[self.data['transfer_condition'] == 'within_config']),
            'transfer_records': len(self.data[self.data['transfer_condition'] != 'within_config']),
            'unique_models': self.data['model_id'].nunique(),
            'config_pairs_available': len(self.data.groupby(['config_L', 'config_m', 'target_config_L', 'target_config_m'])),
            'missing_transfer_conditions': [],
            'incomplete_transfer_models': []
        }
        
        # Check for expected transfer conditions
        expected_conditions = ['within_config', 'cross_L', 'cross_m', 'cross_config']
        available_conditions = set(self.data['transfer_condition'].unique())
        missing = [c for c in expected_conditions if c not in available_conditions]
        validation['missing_transfer_conditions'] = missing
        
        # Check model transfer completeness
        for model_id in self.data['model_id'].unique():
            model_data = self.data[self.data['model_id'] == model_id]
            model_conditions = set(model_data['transfer_condition'].unique())
            if 'within_config' not in model_conditions:
                validation['incomplete_transfer_models'].append(model_id)
        
        return validation
    
    def save_transfer_metrics(self) -> Path:
        """Save transfer metrics to CSV."""
        if not self.transfer_metrics:
            raise ValueError("No transfer metrics to save")
        
        metrics_df = pd.DataFrame([
            {
                'model_id': m.model_id,
                'train_config_L': m.train_config_L,
                'train_config_m': m.train_config_m,
                'test_config_L': m.test_config_L,
                'test_config_m': m.test_config_m,
                'n_train': m.n_train,
                'within_config_accuracy': m.within_config_accuracy,
                'transfer_accuracy': m.transfer_accuracy,
                'transfer_degradation': m.transfer_degradation,
                'transfer_type': m.transfer_type,
                'depth_increase': m.depth_increase,
                'synonym_increase': m.synonym_increase,
                'transfer_success': m.transfer_success,
                'transfer_scaling_slope': m.transfer_scaling_slope
            }
            for m in self.transfer_metrics
        ])
        
        output_path = self.output_dir / 'transfer_metrics.csv'
        metrics_df.to_csv(output_path, index=False)
        
        print(f"Saved transfer metrics: {output_path}")
        return output_path


def run_transfer_analysis(
    phase1_results_path: str | Path,
    output_dir: str | Path
) -> dict[str, t.Any]:
    """Run complete transfer learning analysis pipeline."""
    analyzer = TransferAnalyzer(Path(phase1_results_path), Path(output_dir))
    
    # Load and process data
    analyzer.load_existing_results()
    analyzer.create_transfer_pairs()
    
    # Validate data completeness
    validation = analyzer.validate_transfer_completeness()
    print(f"Data validation: {validation['transfer_records']} transfer records, {validation['unique_models']} models")
    
    # Compute metrics and analyze patterns
    analyzer.compute_transfer_metrics()
    patterns = analyzer.analyze_transfer_patterns()
    
    # Create transfer matrix
    analyzer.create_transfer_matrix()
    
    # Generate outputs
    metrics_path = analyzer.save_transfer_metrics()
    report_path = analyzer.generate_transfer_report()
    
    return {
        'transfer_metrics_path': metrics_path,
        'report_path': report_path,
        'patterns': patterns,
        'validation': validation,
        'total_transfer_pairs_analyzed': len(analyzer.transfer_metrics),
        'transfer_matrix': analyzer.transfer_matrix
    }

In [6]:

# Test function
def test_transfer_analysis() -> None:
    """Test transfer analysis with sample data."""
    # This would use actual Phase 1 results path
    phase1_path = Path("/Users/jliu/workspace/ICL/results/raw/raw_evaluations/icl_performance.parquet")
    output_dir = Path("/Users/jliu/workspace/ICL/results/scaling_analysis_results")
    
    
    if phase1_path.exists():
        results = run_transfer_analysis(phase1_path, output_dir)
        print(f"Transfer analysis completed: {results['total_transfer_pairs_analyzed']} transfer pairs")
    else:
        print("Phase 1 results not found - run Phase 1 evaluation first")


if __name__ == "__main__":
    test_transfer_analysis()

Loaded 6426 evaluation records from Phase 1
Filtered to 2142 normal control sequences
Within-config records: 342
Transfer records: 1800
Created 1800 transfer pairs
Data validation: 1800 transfer records, 6 models
Computed transfer metrics for 1800 transfer pairs
Created transfer matrix: (1, 3)
Saved transfer metrics: /Users/jliu/workspace/ICL/results/scaling_analysis_results/transfer_metrics.csv
Generated transfer report: /Users/jliu/workspace/ICL/results/scaling_analysis_results/transfer_analysis_report.html
Transfer analysis completed: 1800 transfer pairs


# scaling law

In [ ]:
"""RQ2: Scaling Analysis - Context Length Requirements vs Hierarchical Complexity."""

import typing as t
from pathlib import Path
from dataclasses import dataclass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
import warnings


@dataclass
class ScalingMetrics:
    """Metrics for context length scaling analysis."""
    
    model_id: str
    config_L: int
    config_m: int
    n_train: int
    complexity_score: float
    
    # Optimal context sizes for different thresholds
    optimal_context_50: float
    optimal_context_70: float
    optimal_context_90: float
    
    # Scaling law parameters
    exponential_a: float
    exponential_b: float
    exponential_r2: float
    
    power_a: float
    power_b: float
    power_r2: float
    
    log_a: float
    log_b: float
    log_r2: float
    
    # Best fitting model
    best_model: str
    best_r2: float
    
    # Scaling characteristics
    saturation_point: float
    max_accuracy: float
    scaling_efficiency: float


class ScalingAnalyzer:
    """Analyzes context length scaling patterns with hierarchical complexity."""
    
    def __init__(self, phase1_results_path: Path, output_dir: Path):
        """Initialize scaling analyzer with Phase 1 results."""
        self.phase1_results_path = phase1_results_path
        self.output_dir = output_dir
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.data: pd.DataFrame | None = None
        self.filtered_data: pd.DataFrame | None = None
        self.scaling_metrics: list[ScalingMetrics] = []
        
    def load_existing_results(self) -> pd.DataFrame:
        """Load Phase 1 evaluation results for scaling analysis."""
        if not self.phase1_results_path.exists():
            raise FileNotFoundError(f"Phase 1 results not found: {self.phase1_results_path}")
        
        self.data = pd.read_parquet(self.phase1_results_path)
        print(f"Loaded {len(self.data)} evaluation records from Phase 1")
        
        # Filter for within-config normal sequences (clean scaling data)
        self.filtered_data = self.data[
            (self.data['transfer_condition'] == 'within_config') &
            (self.data['control_type'] == 'normal')
        ].copy()
        
        print(f"Filtered to {len(self.filtered_data)} within-config normal sequences")
        return self.filtered_data
    
    def detect_optimal_context_sizes(self) -> dict[str, t.Any]:
        """Detect optimal context sizes for different performance thresholds."""
        if self.filtered_data is None:
            raise ValueError("Must load data first")
        
        thresholds = [0.5, 0.7, 0.9]
        optimal_results = {
            'model_thresholds': [],
            'threshold_statistics': {},
            'complexity_correlations': {}
        }
        
        # Group by model for individual analysis
        model_groups = self.filtered_data.groupby([
            'model_id', 'config_L', 'config_m', 'n_train', 'checkpoint_step'
        ])
        
        for (model_id, config_L, config_m, n_train, checkpoint_step), group in model_groups:
            # Get context-accuracy curve for this model
            context_accuracies = group.groupby('context_size')['accuracy'].mean().sort_index()
            
            if len(context_accuracies) < 3:  # Need sufficient data points
                continue
            
            model_thresholds = {
                'model_id': model_id,
                'config_L': config_L,
                'config_m': config_m,
                'n_train': n_train,
                'complexity_score': config_L * config_m
            }
            
            # Find optimal context size for each threshold
            for threshold in thresholds:
                optimal_k = self._find_optimal_context_size(context_accuracies, threshold)
                model_thresholds[f'optimal_context_{int(threshold*100)}'] = optimal_k
            
            # Additional metrics
            model_thresholds['max_accuracy'] = context_accuracies.max()
            model_thresholds['saturation_point'] = self._find_saturation_point(context_accuracies)
            
            optimal_results['model_thresholds'].append(model_thresholds)
        
        # Compute threshold statistics
        if optimal_results['model_thresholds']:
            threshold_df = pd.DataFrame(optimal_results['model_thresholds'])
            
            for threshold in thresholds:
                thresh_key = f'optimal_context_{int(threshold*100)}'
                valid_thresholds = threshold_df[threshold_df[thresh_key] != float('inf')]
                
                optimal_results['threshold_statistics'][f'threshold_{threshold}'] = {
                    'mean_optimal_k': valid_thresholds[thresh_key].mean() if len(valid_thresholds) > 0 else float('inf'),
                    'std_optimal_k': valid_thresholds[thresh_key].std() if len(valid_thresholds) > 0 else 0,
                    'achievement_rate': len(valid_thresholds) / len(threshold_df),
                    'min_optimal_k': valid_thresholds[thresh_key].min() if len(valid_thresholds) > 0 else float('inf'),
                    'max_optimal_k': valid_thresholds[thresh_key].max() if len(valid_thresholds) > 0 else float('inf')
                }
            
            # Complexity correlations
            for threshold in thresholds:
                thresh_key = f'optimal_context_{int(threshold*100)}'
                valid_data = threshold_df[threshold_df[thresh_key] != float('inf')]
                
                if len(valid_data) > 3:
                    complexity_corr = valid_data['complexity_score'].corr(valid_data[thresh_key])
                    optimal_results['complexity_correlations'][f'threshold_{threshold}'] = complexity_corr
        
        print(f"Analyzed optimal context sizes for {len(optimal_results['model_thresholds'])} models")
        return optimal_results
    
    def _find_optimal_context_size(self, context_accuracies: pd.Series, threshold: float) -> float:
        """Find minimum context size achieving performance threshold."""
        for context_size, accuracy in context_accuracies.items():
            if accuracy >= threshold:
                return float(context_size)
        return float('inf')  # Threshold not achieved
    
    def _find_saturation_point(self, context_accuracies: pd.Series) -> float:
        """Find context size where performance saturates (diminishing returns)."""
        if len(context_accuracies) < 3:
            return float('inf')
        
        # Compute improvements between consecutive context sizes
        improvements = context_accuracies.diff().dropna()
        
        # Find where improvement drops below 5% of max improvement
        max_improvement = improvements.max()
        saturation_threshold = 0.05 * max_improvement
        
        for context_size, improvement in improvements.items():
            if improvement < saturation_threshold:
                return float(context_size)
        
        return float(max(context_accuracies.index))  # No clear saturation
    
    def fit_scaling_laws(self) -> dict[str, t.Any]:
        """Fit different scaling functions to context-performance curves."""
        if self.filtered_data is None:
            raise ValueError("Must load data first")
        
        scaling_results = {
            'model_fits': [],
            'law_comparisons': {},
            'parameter_distributions': {}
        }
        
        # Group by model for curve fitting
        model_groups = self.filtered_data.groupby([
            'model_id', 'config_L', 'config_m', 'n_train', 'checkpoint_step'
        ])
        
        for (model_id, config_L, config_m, n_train, checkpoint_step), group in model_groups:
            context_accuracies = group.groupby('context_size')['accuracy'].mean().sort_index()
            
            if len(context_accuracies) < 4:  # Need sufficient points for fitting
                continue
            
            context_sizes = context_accuracies.index.values
            accuracies = context_accuracies.values
            
            # Fit different scaling laws
            model_fit = self._fit_scaling_laws_single_model(
                context_sizes, accuracies, model_id, config_L, config_m, n_train
            )
            
            if model_fit:
                scaling_results['model_fits'].append(model_fit)
        
        # Analyze law comparisons
        if scaling_results['model_fits']:
            fits_df = pd.DataFrame(scaling_results['model_fits'])
            
            # Compare model performance
            scaling_results['law_comparisons'] = {
                'exponential_wins': (fits_df['best_model'] == 'exponential').sum(),
                'power_wins': (fits_df['best_model'] == 'power').sum(),
                'logarithmic_wins': (fits_df['best_model'] == 'logarithmic').sum(),
                'mean_exponential_r2': fits_df['exponential_r2'].mean(),
                'mean_power_r2': fits_df['power_r2'].mean(),
                'mean_log_r2': fits_df['log_r2'].mean()
            }
            
            # Parameter distributions
            scaling_results['parameter_distributions'] = {
                'exponential_a': fits_df['exponential_a'].describe().to_dict(),
                'exponential_b': fits_df['exponential_b'].describe().to_dict(),
                'power_a': fits_df['power_a'].describe().to_dict(),
                'power_b': fits_df['power_b'].describe().to_dict(),
                'log_a': fits_df['log_a'].describe().to_dict(),
                'log_b': fits_df['log_b'].describe().to_dict()
            }
        
        print(f"Fitted scaling laws for {len(scaling_results['model_fits'])} models")
        return scaling_results
    
    def _fit_scaling_laws_single_model(
        self, context_sizes: np.ndarray, accuracies: np.ndarray,
        model_id: str, config_L: int, config_m: int, n_train: int
    ) -> dict[str, t.Any] | None:
        """Fit scaling laws for a single model."""
        try:
            fits = {}
            
            # 1. Exponential saturation: acc = a * (1 - exp(-b*k))
            try:
                def exponential_func(k, a, b):
                    return a * (1 - np.exp(-b * k))
                
                popt_exp, _ = curve_fit(
                    exponential_func, context_sizes, accuracies,
                    bounds=([0, 0], [1.5, 10]),  # Reasonable bounds
                    maxfev=1000
                )
                pred_exp = exponential_func(context_sizes, *popt_exp)
                r2_exp = r2_score(accuracies, pred_exp)
                
                fits['exponential'] = {
                    'a': popt_exp[0], 'b': popt_exp[1], 'r2': r2_exp
                }
            except:
                fits['exponential'] = {'a': 0, 'b': 0, 'r2': -np.inf}
            
            # 2. Power law: acc = a * k^b
            try:
                def power_func(k, a, b):
                    return a * np.power(k, b)
                
                # Use log-transform for stability
                log_k = np.log(context_sizes)
                log_acc = np.log(np.maximum(accuracies, 1e-10))  # Avoid log(0)
                
                slope, intercept, r_value, _, _ = stats.linregress(log_k, log_acc)
                a_power = np.exp(intercept)
                b_power = slope
                
                pred_power = power_func(context_sizes, a_power, b_power)
                r2_power = r2_score(accuracies, pred_power)
                
                fits['power'] = {
                    'a': a_power, 'b': b_power, 'r2': r2_power
                }
            except:
                fits['power'] = {'a': 0, 'b': 0, 'r2': -np.inf}
            
            # 3. Logarithmic: acc = a * log(k) + b
            try:
                log_context = np.log(context_sizes)
                slope, intercept, r_value, _, _ = stats.linregress(log_context, accuracies)
                
                pred_log = slope * log_context + intercept
                r2_log = r2_score(accuracies, pred_log)
                
                fits['logarithmic'] = {
                    'a': slope, 'b': intercept, 'r2': r2_log
                }
            except:
                fits['logarithmic'] = {'a': 0, 'b': 0, 'r2': -np.inf}
            
            # Determine best model
            best_model = max(fits.keys(), key=lambda k: fits[k]['r2'])
            best_r2 = fits[best_model]['r2']
            
            return {
                'model_id': model_id,
                'config_L': config_L,
                'config_m': config_m,
                'n_train': n_train,
                'complexity_score': config_L * config_m,
                'exponential_a': fits['exponential']['a'],
                'exponential_b': fits['exponential']['b'],
                'exponential_r2': fits['exponential']['r2'],
                'power_a': fits['power']['a'],
                'power_b': fits['power']['b'],
                'power_r2': fits['power']['r2'],
                'log_a': fits['logarithmic']['a'],
                'log_b': fits['logarithmic']['b'],
                'log_r2': fits['logarithmic']['r2'],
                'best_model': best_model,
                'best_r2': best_r2
            }
            
        except Exception as e:
            warnings.warn(f"Failed to fit scaling laws for {model_id}: {e}")
            return None
    
    def analyze_complexity_scaling(self) -> dict[str, t.Any]:
        """Analyze how complexity affects context requirements."""
        if self.filtered_data is None:
            raise ValueError("Must load data first")
        
        complexity_results = {
            'complexity_effects': {},
            'configuration_analysis': {},
            'training_size_effects': {}
        }
        
        # Create comprehensive metrics DataFrame
        model_metrics = []
        
        model_groups = self.filtered_data.groupby([
            'model_id', 'config_L', 'config_m', 'n_train', 'checkpoint_step'
        ])
        
        for (model_id, config_L, config_m, n_train, checkpoint_step), group in model_groups:
            context_accuracies = group.groupby('context_size')['accuracy'].mean().sort_index()
            
            if len(context_accuracies) < 3:
                continue
            
            complexity_score = config_L * config_m
            
            # Compute various scaling metrics
            optimal_50 = self._find_optimal_context_size(context_accuracies, 0.5)
            optimal_70 = self._find_optimal_context_size(context_accuracies, 0.7)
            max_accuracy = context_accuracies.max()
            
            # Scaling efficiency: accuracy per unit context
            scaling_efficiency = max_accuracy / max(context_accuracies.index)
            
            model_metrics.append({
                'model_id': model_id,
                'config_L': config_L,
                'config_m': config_m,
                'n_train': n_train,
                'complexity_score': complexity_score,
                'optimal_context_50': optimal_50,
                'optimal_context_70': optimal_70,
                'max_accuracy': max_accuracy,
                'scaling_efficiency': scaling_efficiency
            })
        
        if not model_metrics:
            return complexity_results
        
        metrics_df = pd.DataFrame(model_metrics)
        
        # Analyze complexity effects
        complexity_results['complexity_effects'] = {
            'complexity_vs_optimal_50': self._safe_correlation(
                metrics_df, 'complexity_score', 'optimal_context_50'
            ),
            'complexity_vs_optimal_70': self._safe_correlation(
                metrics_df, 'complexity_score', 'optimal_context_70'
            ),
            'complexity_vs_max_accuracy': metrics_df['complexity_score'].corr(
                metrics_df['max_accuracy']
            ),
            'complexity_vs_efficiency': metrics_df['complexity_score'].corr(
                metrics_df['scaling_efficiency']
            )
        }
        
        # Configuration dimension analysis
        complexity_results['configuration_analysis'] = {
            'L_effects': self._analyze_dimension_effects(metrics_df, 'config_L'),
            'm_effects': self._analyze_dimension_effects(metrics_df, 'config_m'),
            'interaction_effects': self._analyze_interaction_effects(metrics_df)
        }
        
        # Training size effects
        complexity_results['training_size_effects'] = {
            'training_vs_optimal_50': self._safe_correlation(
                metrics_df, 'n_train', 'optimal_context_50'
            ),
            'training_vs_efficiency': metrics_df['n_train'].corr(
                metrics_df['scaling_efficiency']
            ),
            'training_size_scaling': self._analyze_training_size_scaling(metrics_df)
        }
        
        return complexity_results
    
    def _safe_correlation(self, df: pd.DataFrame, col1: str, col2: str) -> float:
        """Compute correlation safely, handling infinite values."""
        valid_data = df[(df[col1] != float('inf')) & (df[col2] != float('inf'))]
        if len(valid_data) < 3:
            return 0.0
        return valid_data[col1].corr(valid_data[col2])
    
    def _analyze_dimension_effects(self, metrics_df: pd.DataFrame, dimension: str) -> dict[str, float]:
        """Analyze effects of a single configuration dimension."""
        effects = {}
        
        for metric in ['optimal_context_50', 'optimal_context_70', 'max_accuracy', 'scaling_efficiency']:
            correlation = self._safe_correlation(metrics_df, dimension, metric)
            effects[f'{dimension}_vs_{metric}'] = correlation
        
        return effects
    
    def _analyze_interaction_effects(self, metrics_df: pd.DataFrame) -> dict[str, float]:
        """Analyze L×m interaction effects."""
        interactions = {}
        
        # Create interaction term
        metrics_df['L_m_interaction'] = metrics_df['config_L'] * metrics_df['config_m']
        
        for metric in ['optimal_context_50', 'max_accuracy', 'scaling_efficiency']:
            correlation = self._safe_correlation(metrics_df, 'L_m_interaction', metric)
            interactions[f'L_m_interaction_vs_{metric}'] = correlation
        
        return interactions
    
    def _analyze_training_size_scaling(self, metrics_df: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze how training size affects scaling patterns."""
        scaling_by_size = {}
        
        for n_train in sorted(metrics_df['n_train'].unique()):
            size_data = metrics_df[metrics_df['n_train'] == n_train]
            
            if len(size_data) > 2:
                scaling_by_size[n_train] = {
                    'mean_optimal_50': size_data['optimal_context_50'].replace(
                        float('inf'), np.nan
                    ).mean(),
                    'mean_max_accuracy': size_data['max_accuracy'].mean(),
                    'mean_efficiency': size_data['scaling_efficiency'].mean(),
                    'count': len(size_data)
                }
        
        return scaling_by_size
    
    def compare_scaling_patterns(self) -> dict[str, t.Any]:
        """Compare scaling patterns across configurations."""
        if self.filtered_data is None:
            raise ValueError("Must load data first")
        
        comparison_results = {
            'fixed_L_analysis': {},
            'fixed_m_analysis': {},
            'joint_scaling_analysis': {}
        }
        
        # Fixed L, varying m analysis
        for L in sorted(self.filtered_data['config_L'].unique()):
            L_data = self.filtered_data[self.filtered_data['config_L'] == L]
            comparison_results['fixed_L_analysis'][f'L_{L}'] = self._analyze_fixed_dimension_scaling(
                L_data, 'config_m'
            )
        
        # Fixed m, varying L analysis  
        for m in sorted(self.filtered_data['config_m'].unique()):
            m_data = self.filtered_data[self.filtered_data['config_m'] == m]
            comparison_results['fixed_m_analysis'][f'm_{m}'] = self._analyze_fixed_dimension_scaling(
                m_data, 'config_L'
            )
        
        # Joint L,m scaling patterns
        comparison_results['joint_scaling_analysis'] = self._analyze_joint_scaling_patterns()
        
        return comparison_results
    
    def _analyze_fixed_dimension_scaling(
        self, data: pd.DataFrame, varying_dimension: str
    ) -> dict[str, t.Any]:
        """Analyze scaling when one dimension is fixed."""
        scaling_analysis = {}
        
        for dim_value in sorted(data[varying_dimension].unique()):
            dim_data = data[data[varying_dimension] == dim_value]
            
            # Aggregate across models with same configuration
            config_scaling = dim_data.groupby('context_size')['accuracy'].mean()
            
            if len(config_scaling) >= 3:
                # Compute scaling characteristics
                max_acc = config_scaling.max()
                optimal_50 = self._find_optimal_context_size(config_scaling, 0.5)
                
                scaling_analysis[f'{varying_dimension}_{dim_value}'] = {
                    'max_accuracy': max_acc,
                    'optimal_context_50': optimal_50,
                    'context_scaling_slope': self._compute_scaling_slope(config_scaling)
                }
        
        return scaling_analysis
    
    def _compute_scaling_slope(self, context_accuracies: pd.Series) -> float:
        """Compute linear scaling slope for context-accuracy relationship."""
        if len(context_accuracies) < 2:
            return 0.0
        
        context_sizes = context_accuracies.index.values
        accuracies = context_accuracies.values
        
        try:
            slope, _, _, _, _ = stats.linregress(context_sizes, accuracies)
            return slope
        except:
            return 0.0
    
    def _analyze_joint_scaling_patterns(self) -> dict[str, t.Any]:
        """Analyze joint L,m scaling patterns."""
        joint_patterns = {}
        
        # Group by configuration
        config_groups = self.filtered_data.groupby(['config_L', 'config_m'])
        
        for (L, m), config_data in config_groups:
            if len(config_data) < 10:  # Need sufficient data
                continue
            
            # Average across models with same configuration
            config_scaling = config_data.groupby('context_size')['accuracy'].mean()
            
            if len(config_scaling) >= 3:
                complexity = L * m
                max_acc = config_scaling.max()
                optimal_50 = self._find_optimal_context_size(config_scaling, 0.5)
                slope = self._compute_scaling_slope(config_scaling)
                
                joint_patterns[f'L{L}_m{m}'] = {
                    'complexity': complexity,
                    'max_accuracy': max_acc,
                    'optimal_context_50': optimal_50,
                    'scaling_slope': slope,
                    'config_L': L,
                    'config_m': m
                }
        
        return joint_patterns
    
    def compute_scaling_metrics(self) -> list[ScalingMetrics]:
        """Compute comprehensive scaling metrics combining all analyses."""
        if self.filtered_data is None:
            raise ValueError("Must load data first")
        
        # Get results from individual analyses
        optimal_results = self.detect_optimal_context_sizes()
        scaling_laws = self.fit_scaling_laws()
        
        # Combine into comprehensive metrics
        self.scaling_metrics = []
        
        # Create lookup dictionaries
        optimal_lookup = {
            (row['model_id']): row for row in optimal_results['model_thresholds']
        }
        
        scaling_lookup = {
            (row['model_id']): row for row in scaling_laws['model_fits']
        }
        
        # Merge data
        all_model_ids = set(optimal_lookup.keys()) | set(scaling_lookup.keys())
        
        for model_id in all_model_ids:
            optimal_data = optimal_lookup.get(model_id, {})
            scaling_data = scaling_lookup.get(model_id, {})
            
            if not optimal_data or not scaling_data:
                continue  # Skip incomplete data
            
            # Calculate scaling efficiency
            max_acc = optimal_data.get('max_accuracy', 0)
            max_context = max(self.filtered_data['context_size'])
            scaling_efficiency = max_acc / max_context if max_context > 0 else 0
            
            metrics = ScalingMetrics(
                model_id=model_id,
                config_L=optimal_data.get('config_L', 0),
                config_m=optimal_data.get('config_m', 0),
                n_train=optimal_data.get('n_train', 0),
                complexity_score=optimal_data.get('complexity_score', 0),
                
                optimal_context_50=optimal_data.get('optimal_context_50', float('inf')),
                optimal_context_70=optimal_data.get('optimal_context_70', float('inf')),
                optimal_context_90=optimal_data.get('optimal_context_90', float('inf')),
                
                exponential_a=scaling_data.get('exponential_a', 0),
                exponential_b=scaling_data.get('exponential_b', 0),
                exponential_r2=scaling_data.get('exponential_r2', 0),
                
                power_a=scaling_data.get('power_a', 0),
                power_b=scaling_data.get('power_b', 0),
                power_r2=scaling_data.get('power_r2', 0),
                
                log_a=scaling_data.get('log_a', 0),
                log_b=scaling_data.get('log_b', 0),
                log_r2=scaling_data.get('log_r2', 0),
                
                best_model=scaling_data.get('best_model', 'none'),
                best_r2=scaling_data.get('best_r2', 0),
                
                saturation_point=optimal_data.get('saturation_point', float('inf')),
                max_accuracy=optimal_data.get('max_accuracy', 0),
                scaling_efficiency=scaling_efficiency
            )
            
            self.scaling_metrics.append(metrics)
        
        print(f"Computed comprehensive scaling metrics for {len(self.scaling_metrics)} models")
        return self.scaling_metrics
    
    def generate_scaling_report(self) -> Path:
        """Generate comprehensive scaling analysis report."""
        report_path = self.output_dir / 'scaling_analysis_report.html'
        
        # Create visualizations
        self._create_scaling_visualizations()
        
        # Generate HTML report
        html_content = self._generate_html_report()
        
        with open(report_path, 'w') as f:
            f.write(html_content)
        
        print(f"Generated scaling report: {report_path}")
        return report_path
    
    def _create_scaling_visualizations(self) -> None:
        """Create comprehensive scaling analysis visualizations."""
        if not self.scaling_metrics:
            return
        
        # Create comprehensive visualization grid
        plt.figure(figsize=(20, 16))
        
        # 1. Optimal context size vs complexity
        plt.subplot(4, 4, 1)
        complexities = [m.complexity_score for m in self.scaling_metrics]
        optimal_50s = [m.optimal_context_50 if m.optimal_context_50 != float('inf') else np.nan 
                      for m in self.scaling_metrics]
        
        plt.scatter(complexities, optimal_50s, alpha=0.6)
        plt.xlabel('Configuration Complexity (L×m)')
        plt.ylabel('Optimal Context Size (50% threshold)')
        plt.title('Context Requirements vs Complexity')
        
        # 2. Scaling law comparison
        plt.subplot(4, 4, 2)
        exp_r2s = [m.exponential_r2 for m in self.scaling_metrics]
        power_r2s = [m.power_r2 for m in self.scaling_metrics]
        log_r2s = [m.log_r2 for m in self.scaling_metrics]
        
        plt.boxplot([exp_r2s, power_r2s, log_r2s], 
                   labels=['Exponential', 'Power', 'Logarithmic'])
        plt.ylabel('R² Score')
        plt.title('Scaling Law Fit Comparison')
        
        # 3. Best model distribution
        plt.subplot(4, 4, 3)
        best_models = [m.best_model for m in self.scaling_metrics]
        model_counts = {model: best_models.count(model) for model in set(best_models)}
        plt.bar(model_counts.keys(), model_counts.values())
        plt.ylabel('Number of Models')
        plt.title('Best Fitting Scaling Law')
        plt.xticks(rotation=45)
        
        # 4. Scaling efficiency vs complexity
        plt.subplot(4, 4, 4)
        

        
                        